# 1. Configurando as dependências e o MLFlow

In [1]:
import sys
import warnings
from pathlib import Path
import importlib

import pandas as pd
import mlflow
from mlflow.models import infer_signature
from mlflow.sklearn import log_model

# Configurações para ignorar warnings
warnings.filterwarnings('ignore')

# Define a pasta raiz do repo e caso não ache a pasta src lá, adiciona a mesma
repo_root = Path("..").resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Recarrega o módulo tracking para refletir mudanças recentes de assinatura
import tracking
importlib.reload(tracking)

# Funções customizadas para configurar o MLflow Tracking e obter informações do ambiente
from tracking import (
    build_default_run_tags,
    configure_mlflow_tracking,
    get_owner,
    normalize_params,
 )

EXPERIMENT_NAME = "stg_notebook_training"
RANDOM_SEED = 42

MLFLOW_DB_PATH = repo_root / "mlflow.db"
MLFLOW_ARTIFACTS_PATH = repo_root / "data" / "mlflow_artifacts"
EXPERIMENT_TAGS = {
    "project": "ml-churn-prediction",
    "business_domain": "telecom",
    "problem_type": "binary_classification",
    "target": "churn",
    # ROC_AUC mede a capacidade do modelo em classificar corretamente os clientes 
    # que irão "churnar". É agnóstica ao threshold, boa para datasets desbalanceados.
    "primary_metric": "roc_auc",
    "dataset_name": "telco_customer_churn",
    "owner": get_owner(),
}

tracking_uri = configure_mlflow_tracking(
    experiment_name=EXPERIMENT_NAME,
    db_path=MLFLOW_DB_PATH,
    experiment_tags=EXPERIMENT_TAGS,
    artifact_root_path=MLFLOW_ARTIFACTS_PATH
)

print(f"\nMLflow Tracking configurado com sucesso!")
print("Tracking URI:", tracking_uri)
print("Experimento:", EXPERIMENT_NAME)
print("Artifacts root:", MLFLOW_ARTIFACTS_PATH)

c:\Users\gustavo.dellanhol\Desktop\ml-churn-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/03/24 14:43:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/24 14:43:19 INFO mlflow.store.db.utils: Updating database tables



MLflow Tracking configurado com sucesso!
Tracking URI: sqlite:///C:/Users/gustavo.dellanhol/Desktop/ml-churn-prediction/mlflow.db
Experimento: stg_notebook_training
Artifacts root: C:\Users\gustavo.dellanhol\Desktop\ml-churn-prediction\data\mlflow_artifacts


## 1.1 Criando a função para treinar o modelo, calcular as métricas e logar tudo no MLFlow

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_and_log_model(model, run_name, stage, model_family, 
                        sampling_strategy, X_tr, y_tr, X_te, y_te,
                        model_params=None, extra_tags=None):
    """Treina modelo, calcula métricas e loga no MLflow"""
    
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags(
            build_default_run_tags(
                stage=stage,
                model_family=model_family,
                sampling_strategy=sampling_strategy,
                extra_tags=extra_tags or {},
            )
        )
        
        if model_params:
            mlflow.log_params(normalize_params(model_params))
            
        model.fit(X_tr, y_tr)

        # Log do modelo treinado como artefato no MLflow
        input_example = X_tr.head(5) if hasattr(X_tr, "head") else X_tr[:5]
        predictions_example = model.predict(input_example)
        signature = infer_signature(input_example, predictions_example)
        log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=input_example,
            serialization_format="skops",
        )

        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)
        
        metrics = {
            "train_accuracy": accuracy_score(y_tr, y_pred_train),
            "test_accuracy": accuracy_score(y_te, y_pred_test),
            "test_f1_score": f1_score(y_te, y_pred_test),
            "test_precision": precision_score(y_te, y_pred_test),
            "test_recall": recall_score(y_te, y_pred_test),
            "overfitting": accuracy_score(y_tr, y_pred_train) - accuracy_score(y_te, y_pred_test),
        }
        
        mlflow.log_metrics(metrics)
        print(f"=== {run_name.upper()} ===")
        for key, val in metrics.items():
            print(f"{key}: {val:.4f}")

# 2. Setando o baseline do projeto e configurando pela primeira vez o experimento do MLFlow

### 2.1 One-hot Encoding

A fim de facilitar a ingestão nos modelos de ML, utilizamos o __get_dummies__ do pandas para fazer o encoding do dataset

In [3]:
df_clean = pd.read_csv("../data/processed/telco_customer_churn_eda_pre-processed.csv")
print("Shape do dataset:", df_clean.shape)


# One-hot encoding das variáveis categóricas relevantes
categorical_cols = ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 
                    'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 
                    'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 
                    'Payment Method']
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

# Converter booleanos para inteiro
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"Shape após one-hot encoding: {df_encoded.shape}")
df_encoded.head()

# Salva uma cópia do dataset
df_encoded.to_csv("../data/processed/telco_customer_churn_eda_encoded.csv", index=False)

Shape do dataset: (7043, 23)
Shape após one-hot encoding: (7043, 34)


### 2.2 Separando o dataset entre treino e teste

In [4]:
from sklearn.model_selection import train_test_split

# Dividindo o dataset em features (X) e target (y)
X = df_encoded.drop(columns=['target', 'CLTV', 'Total Charges'])
y = df_encoded['target']

# Divida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_SEED)

### 2.3 Treinando o baseline nos dados do EDA

DummyClassifier

In [15]:
from sklearn.dummy import DummyClassifier

dummy_classifier_baseline_model = DummyClassifier(random_state=RANDOM_SEED)

train_and_log_model(
    model = dummy_classifier_baseline_model,
    X_tr = X_train,
    y_tr = y_train,
    X_te = X_test,
    y_te = y_test,
    run_name = "dummy_classifier_baseline",
    stage = "baseline",
    model_family = "dummy_classifier",
    sampling_strategy = "none",
    model_params={"model": "dummy_classifier", "strategy": "most_frequent", "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "defining_baselines"} # Edite aqui para definir novas etapas de runs
)

=== DUMMY_CLASSIFIER_BASELINE ===
train_accuracy: 0.7393
test_accuracy: 0.7161
test_f1_score: 0.0000
test_precision: 0.0000
test_recall: 0.0000
overfitting: 0.0232


Regressão Logística

In [16]:
log_reg_baseline_model = LogisticRegression(random_state = RANDOM_SEED, max_iter = 1000)

train_and_log_model(
    model = log_reg_baseline_model,
    X_tr = X_train,
    y_tr = y_train,
    X_te = X_test,
    y_te = y_test,
    run_name = "logistic_regression_baseline",
    stage = "baseline",
    model_family = "logistic_regression",
    sampling_strategy = "none",
    model_params={"model": "logistic_regression", "max_iter": 1000, "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "defining_baselines"} # Edite aqui para definir novas etapas de runs
)

=== LOGISTIC_REGRESSION_BASELINE ===
train_accuracy: 0.8147
test_accuracy: 0.8027
test_f1_score: 0.6233
test_precision: 0.6805
test_recall: 0.5750
overfitting: 0.0120


## 3. Testando parâmetros diferentes no modelo

In [17]:
log_reg_balanced_model = LogisticRegression(random_state = RANDOM_SEED, max_iter = 1000, class_weight = "balanced")

train_and_log_model(
    model = log_reg_balanced_model,
    X_tr = X_train,
    y_tr = y_train,
    X_te = X_test,
    y_te = y_test,
    run_name = "logistic_regression_balanced",
    stage = "baseline",
    model_family = "logistic_regression",
    sampling_strategy = "class_weight_balanced",
    model_params={"model": "logistic_regression", 'class_weight': "balanced", "max_iter": 1000, "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "testing_different_parameters"} # Edite aqui para definir novas etapas de runs
)

=== LOGISTIC_REGRESSION_BALANCED ===
train_accuracy: 0.7632
test_accuracy: 0.7502
test_f1_score: 0.6444
test_precision: 0.5407
test_recall: 0.7975
overfitting: 0.0130


Devido ao desbalanceamento das classes (73% vs 27%), foi aplicada a estratégia de `class_weight='balanced'` na Regressão Logística, visando melhorar a capacidade do modelo em identificar clientes com churn. A abordagem resultou em aumento do recall, tornando o modelo mais sensível à classe minoritária.

| Métrica     | Baseline | Balanced | Mudança       |
| ----------- | -------- | -------- | ------------- |
| Accuracy    | 0.8098   | 0.7551   | ⬇️ (esperado) |
| Precision   | 0.7063   | 0.5459   | ⬇️            |
| Recall      | 0.5650   | 0.8175   | 🚀⬆️ MUITO    |
| F1 Score    | 0.6278   | 0.6547   | ⬆️            |
| Overfitting | 0.0072   | 0.0065   | ✅ ok         |

O modelo baseline apresentou bom desempenho geral, porém com recall limitado na identificação de churn.
Após aplicar class_weight='balanced', houve um aumento significativo no recall (de 56% para 81%), tornando o modelo mais eficaz na detecção de clientes propensos a churn, ainda que com redução na precisão — um trade-off aceitável para o contexto de negócio.

### 3.2 Utilizando SMOTE nos dados de treino

In [18]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = RANDOM_SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train) # type: ignore

log_reg_balanced_smote_model = LogisticRegression(random_state = RANDOM_SEED, max_iter = 1000)

train_and_log_model(
    model = log_reg_balanced_smote_model,
    X_tr = X_train_smote,
    y_tr = y_train_smote,
    X_te = X_test,
    y_te = y_test,
    run_name = "logistic_regression_balanced_smote",
    stage = "baseline",
    model_family = "logistic_regression",
    sampling_strategy = "smote",
    model_params={"model": "logistic_regression", "max_iter": 1000, "seed": RANDOM_SEED},
    extra_tags = {"run_purpose": "testing_different_parameters"} # Edite aqui para definir novas etapas de runs
)

=== LOGISTIC_REGRESSION_BALANCED_SMOTE ===
train_accuracy: 0.8339
test_accuracy: 0.7729
test_f1_score: 0.6372
test_precision: 0.5830
test_recall: 0.7025
overfitting: 0.0610


| Modelo   | Precision | Recall   | F1       | Overfitting  |
| -------- | --------- | -------- | -------- | ------------ |
| Baseline | 0.70      | 0.56     | 0.62     | 0.007        |
| Balanced | 0.55      | **0.81** | **0.65** | 0.006        |
| SMOTE    | 0.58      | 0.69     | 0.63     | **0.069 ⚠️** |

Foram testadas diferentes estratégias para lidar com desbalanceamento, incluindo class_weight e SMOTE. A abordagem com class_weight='balanced' apresentou melhor desempenho, com maior recall e menor overfitting, sendo mais adequada para o problema de churn.